# **3) Evaluate & Plot**
Purpose : Build evaluation tables and figures for the multi-label genre
          model. Figures are saved to OUTPUT/ and also shown on screen.

Inputs  :
*   OUTPUT/test_scores.npz
*   OUTPUT/model/*.joblib
*   OUTPUT/metrics_summary.json



Outputs :
*   OUTPUT/classification_report_by_genre.csv
*   OUTPUT/top_words_per_genre.csv
*   OUTPUT/fig_f1_by_genre.png
*   OUTPUT/fig_top_prediction_vs_true_genre.png
*   OUTPUT/fig_top_words_per_genre.png
*   OUTPUT/fig_accuracy_measures.png



Packages: pandas, numpy, scikit-learn, matplotlib, joblib

Run     : python SCRIPTS/03_evaluate_and_plot.py   (from the repo root)

Order   : Run AFTER 2. Train Model.py

In [ ]:
# Anthropic. (2026). Claude Sonnet 5 [Large language model]. https://claude.ai/ Artificial intelligence was used to assist with code development and debugging.
import json
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report

OUTPUT_DIR = Path("OUTPUT")
MODEL_DIR = OUTPUT_DIR / "model"
TOP_N_HEATMAP = 10     # genres shown in the true-vs-predicted heatmap (most frequent)
TOP_N_WORDS = 8        # top predictive words listed per genre
N_WORD_PANELS = 4      # number of genres shown in the top-words figure
SHOW_PLOTS = True      # True = display plots on screen too; False = only save


def save_and_show(fig, filename, **kwargs):
    """Save a figure to OUTPUT/, then optionally display it, then close it."""
    fig.savefig(OUTPUT_DIR / filename, dpi=200, **kwargs)  # save BEFORE show
    if SHOW_PLOTS:
        plt.show()
    plt.close(fig)


# Load
scores = np.load(OUTPUT_DIR / "test_scores.npz", allow_pickle=True)
y_true, y_prob, y_pred = scores["y_true"], scores["y_prob"], scores["y_pred"]
classes = scores["classes"]

vectorizer = joblib.load(MODEL_DIR / "tfidf_vectorizer.joblib")
clf = joblib.load(MODEL_DIR / "logreg_model.joblib")
with open(OUTPUT_DIR / "metrics_summary.json") as f:
    summary = json.load(f)
results = summary["test_results"]

# PER-GENRE METRICS
# Precision, recall, F1 for each genre treated as its own yes/no question.
# Support = number of test books that truly have the genre.
report = classification_report(y_true, y_pred, target_names=classes,
                               output_dict=True, zero_division=0)
report_df = pd.DataFrame(report).T
report_df.to_csv(OUTPUT_DIR / "classification_report_by_genre.csv")

genre_rows = report_df.drop(index=["micro avg", "macro avg", "weighted avg", "samples avg"])
genre_rows = genre_rows.sort_values("f1-score")

#FIGURE 1: F1 BY GENRE
fig, ax = plt.subplots(figsize=(8, max(4, 0.3 * len(genre_rows))))
ax.barh(genre_rows.index, genre_rows["f1-score"], color="steelblue")
ax.set_xlabel("F1 score (test set)")
ax.set_title("Multi-label Logistic Regression: F1 Score by Genre")
ax.set_xlim(0, 1)
plt.tight_layout()
save_and_show(fig, "fig_f1_by_genre.png")

# FIGURE 2: ACCURACY MEASURES
# Compares the different ways of measuring "accuracy" for multi-label data
measures = {
    "Baseline\n(always most\ncommon genre)": summary["baseline_hit_rate"],
    "Exact match\n(full genre set\nmust match)": results["exact_match_accuracy"],
    "Hit rate\n(top guess is\na true genre)": results["hit_rate_top1"],
    "Top-3 hit rate\n(true genre in\ntop 3 guesses)": results["hit_rate_top3"],
}
fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(list(measures.keys()), list(measures.values()),
              color=["gray", "indianred", "steelblue", "seagreen"])
ax.axhline(0.85, color="black", linestyle="--", linewidth=1, label="85% goal")
for bar, val in zip(bars, measures.values()):
    ax.text(bar.get_x() + bar.get_width() / 2, val + 0.01, f"{val:.2f}",
            ha="center", fontsize=10)
ax.set_ylim(0, 1)
ax.set_ylabel("Share of test books")
ax.set_title("Accuracy Measures vs. 85% Goal (test set)")
ax.legend()
plt.tight_layout()
save_and_show(fig, "fig_accuracy_measures.png")

# TOP PREDICTION VS TRUE GENRE
# For each true genre (row), where does the model's top guess land?
# Row values sum to 1 across ALL genres; only the most common are displayed.
top_pred_idx = np.argmax(y_prob, axis=1)
support = y_true.sum(axis=0)
top_true_idx = np.argsort(support)[::-1][:TOP_N_HEATMAP]

matrix = np.zeros((len(top_true_idx), len(top_true_idx)))
for r, true_g in enumerate(top_true_idx):
    books = y_true[:, true_g] == 1                       # books that have this genre
    for c, pred_g in enumerate(top_true_idx):
        matrix[r, c] = (top_pred_idx[books] == pred_g).mean()

labels = classes[top_true_idx]
fig, ax = plt.subplots(figsize=(9, 8))
im = ax.imshow(matrix, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(len(labels)))
ax.set_yticks(range(len(labels)))
ax.set_xticklabels(labels, rotation=45, ha="right")
ax.set_yticklabels(labels)
ax.set_xlabel("Model's top predicted genre")
ax.set_ylabel("True genre (book has this genre)")
ax.set_title(f"Top Prediction vs. True Genre (top {TOP_N_HEATMAP} genres)")
for i in range(len(labels)):
    for j in range(len(labels)):
        ax.text(j, i, f"{matrix[i, j]:.2f}", ha="center", va="center",
                color="white" if matrix[i, j] > 0.5 else "black", fontsize=8)
fig.colorbar(im, ax=ax, label="Share of books with the true genre")
plt.tight_layout()
save_and_show(fig, "fig_top_prediction_vs_true_genre.png")

# TOP WORDS
# Largest positive coefficients = words/phrases pushing the model toward a genre
feature_names = np.array(vectorizer.get_feature_names_out())
coefs = np.vstack([est.coef_[0] for est in clf.estimators_])  # one row per genre
rows = []
for idx, genre in enumerate(classes):
    top_idx = np.argsort(coefs[idx])[::-1][:TOP_N_WORDS]
    for rank, fi in enumerate(top_idx, start=1):
        rows.append({"genre": genre, "rank": rank, "term": feature_names[fi],
                     "coefficient": round(float(coefs[idx][fi]), 3)})
words_df = pd.DataFrame(rows)
words_df.to_csv(OUTPUT_DIR / "top_words_per_genre.csv", index=False)

plot_genres = classes[top_true_idx][:N_WORD_PANELS]
fig, axes = plt.subplots(1, len(plot_genres), figsize=(4.5 * len(plot_genres), 4))
for ax, genre in zip(np.atleast_1d(axes), plot_genres):
    sub = words_df[words_df["genre"] == genre].sort_values("coefficient")
    ax.barh(sub["term"], sub["coefficient"], color="darkorange")
    ax.set_title(genre, fontsize=10)
    ax.set_xlabel("Coefficient")
plt.suptitle("Most Predictive Terms by Genre", y=1.02)
plt.tight_layout()
save_and_show(fig, "fig_top_words_per_genre.png", bbox_inches="tight")

# PRINT SUMMARY
print("Saved evaluation tables and figures to OUTPUT/\n")
print("Overall test results:")
for name, val in results.items():
    print(f"  {name:22s}: {val:.4f}")
print(f"  {'baseline_hit_rate':22s}: {summary['baseline_hit_rate']:.4f}\n")
print(genre_rows[["precision", "recall", "f1-score", "support"]].round(3).to_string())

Loaded 6609 books.
After genre filtering: 6609 books, 28 genres
Average genres per book: 1.70
Label matrix shape: (6609, 28)
Train size: 5287
Test size: 1322
Fitting 5 folds for each of 12 candidates, totalling 60 fits

Best CV Micro F1: 0.6439
Best settings:
  clf__estimator__C: 5.0
  tfidf__max_features: 20000
  tfidf__ngram_range: (1, 2)

----- TEST SET RESULTS -----
hit_rate_top1           : 0.7542
hit_rate_top3           : 0.8994
exact_match_accuracy    : 0.3896
f1_micro                : 0.6583
f1_macro                : 0.5153
f1_samples              : 0.6559
hamming_loss            : 0.0403
baseline_hit_rate       : 0.3200 (always predicts 'Science Fiction')

Saved models, predictions, cross-validation results, and metrics to OUTPUT/
